# Auto Inference Acceleration benchmark(boto3)


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)

    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]

    while status == "Creating":
        time.sleep(sleep_time)

        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]

        clear_output(wait=True)
        progress += ind
        print(progress)

    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)

    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]

    while status == "Creating":
        time.sleep(sleep_time)

        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]

        clear_output(wait=True)
        progress += ind
        print(progress)

    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.g6e.12xlarge", "num_gpu": 4}
model_id = "Qwen/Qwen3.5-35B-A3B"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

### LMI

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi23.0.0-cu129"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": model_id,
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": json.dumps(instance["num_gpu"]),
    "OPTION_MAX_MODEL_LEN": "32736",
}

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            #"InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1",
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

In [ ]:
ic_name = f"ic-{model_name}"

min_memory_required_in_mb = 10*1024
number_of_accelerator_devices_required = instance["num_gpu"]

_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": min_memory_required_in_mb,
            "NumberOfAcceleratorDevicesRequired": number_of_accelerator_devices_required,
        },
    },
    RuntimeConfig={
        "CopyCount": 1,
    },
)
_ = wait_for_ic(ic_name)

In [ ]:
payload={
    "messages": [
        #{"role": "user", "content": "Name popular places to visit in London?"}
        {"role": "user", "content": "Solve this problem step by step: What is 15% of 240?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"]
print(f'-----------------------\n{usage}')

## Cleanup

In [ ]:
#_ = sm.delete_inference_component(InferenceComponentName=ic_name)

In [ ]:
#_ = sm.delete_endpoint(EndpointName=endpoint_name)
#_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
#_ = sm.delete_model(ModelName=model_name)

## Lumen setup

In [ ]:
import boto3
import json
from botocore.loaders import Loader

# Load the custom service model into botocore
loader = Loader()
api = json.load(open("./sagemaker-2017-07-24.normal.json"))
session = boto3.Session(region_name="us-west-2")

# Create client with custom service model
client = session.client(
    "sagemaker",
    endpoint_url=f"https://api.sagemaker.{region}.amazonaws.com",
    api_version=api.get("metadata", {}).get("apiVersion")
)

In [ ]:
config_name = "ds-c10"

workload_spec = {
   "benchmark": {
        "type": "aiperf"
    },
    "parameters": {
        "public_dataset": "sharegpt",
        "prompt_input_tokens_mean": 500,
        "prompt_input_tokens_stddev": 10,
        "output_tokens_mean": 500,
        "output_tokens_stddev": 10,
        "extra_inputs": "ignore_eos:true",
        "concurrency": 10,
        "request_count": 300,
    },
}

response = client.create_ai_workload_config(
   AIWorkloadConfigName=config_name,
   #DatasetConfig={ # only required if using a custom dataset and/or explicit tokenizer in S3
   #    "InputDataConfig": [
   #        {
   #            "ChannelName": "tokenizer",
   #            "DataSource": {
   #                 "S3DataSource": {
   #                 "S3DataType": "S3Prefix",
   #                 "S3Uri": "s3://sagemaker-us-west-2-736221153822/Qwen/Qwen3-4B/",
   #                 "S3DataDistributionType": "FullyReplicated"
   #                 }
   #             },
   #            "ContentType": "application/octet-stream",
   #        },
   #    ]
   #},
   AIWorkloadConfigs={
       "WorkloadSpec": {"Inline": json.dumps(workload_spec)}
   },
)

print(response["AIWorkloadConfigArn"])

In [ ]:
job_name = "ds-c10"

job_response = client.create_ai_benchmark_job(
    AIBenchmarkJobName=job_name,
    BenchmarkTarget={
        "Endpoint": {
            "Identifier": endpoint_name,
            "InferenceComponents": [ # optional, for IC-endpoints
                {
                    "Identifier": ic_name
                },
            ]
        }
    },
    OutputConfig={
        "S3OutputLocation": "s3://sagemaker-us-west-2-736221153822/benchmark-output/"
    },
    AIWorkloadConfigIdentifier=config_name,
    RoleArn="arn:aws:iam::736221153822:role/SageMaker-Benchmark-Role",
)

print(job_response["AIBenchmarkJobArn"])

In [ ]:
import time

while True:
    response = client.describe_ai_benchmark_job(
        AIBenchmarkJobName=job_name
    )
    status = response["AIBenchmarkJobStatus"]
    print(f"Status: {status}")

    if status in ("Completed", "Failed", "Stopped"):
        break

    time.sleep(30)

if status == "Completed":
    print("Benchmark completed successfully!")
    print(f"Results at: {response['OutputConfig']['S3OutputLocation']}")
elif status == "Failed":
    print(f"Benchmark failed: {response.get('FailureReason', 'unknown')}")

In [ ]:
# Delete a completed/failed/stopped job
client.delete_ai_benchmark_job(AIBenchmarkJobName=job_name)

In [ ]:
# Delete a config
client.delete_ai_workload_config(AIWorkloadConfigName=config_name)